## Reads your raw files from the Volume, runs AI validation on city + country,

**What this notebook does:**
1. Reads raw airport data (`airports.dat.txt`) from Unity Catalog Volume
2. Loads a clean reference list from `cities500.txt` (GeoNames)
3. Applies an LLM-backed Spark UDF to validate `city` and `country` columns
4. Routes rows into 3 Silver Delta tables based on AI confidence score

**Output tables:**
```
geo_assist.cleaned.silver_clean       → confidence ≥ 0.85  (auto-corrected)
geo_assist.cleaned.silver_review      → confidence 0.60–0.84 (human QA queue)
geo_assist.cleaned.silver_quarantine  → confidence < 0.60  (manual lookup)
```

> imports

In [0]:
import time  

from pyspark.sql import functions as F
from pyspark.sql.functions import udf
from pyspark.sql.types import (
    DoubleType, FloatType, IntegerType, StringType, StructField, StructType
)

> config

In [0]:
# ── Unity Catalog destination 
CATALOG = "geo_assist"    # top-level catalog
SCHEMA  = "cleaned"       # schema

# ── Source files (already in your Volume) ─────────────────────────────────────
VOLUME         = "/Volumes/geo_assist/raw/geo_assist"
AIRPORTS_FILE  = f"{VOLUME}/airports.dat.txt"   # dirty raw data
REFERENCE_FILE = f"{VOLUME}/cities500.txt"       # GeoNames clean reference

# Databricks Foundation Models 
# Billed per token against your Databricks account.
LLM_MODEL = "databricks-meta-llama-3-3-70b-instruct"


# Tune these based on your project's risk tolerance:
AUTO_ACCEPT_THRESHOLD   = 0.85   # >= 0.85 → silver_clean   (trusted, auto-corrected)
MANUAL_REVIEW_THRESHOLD = 0.60   # >= 0.60 → silver_review  (uncertain, needs human)
                                  # <  0.60 → silver_quarantine (too uncertain)

# How many clean city/country examples to pass to the LLM per call.
# More examples = better accuracy, but longer prompt = more tokens = higher cost.
REFERENCE_SAMPLE_SIZE = 15

# RETRY_SLEEP_SECONDS: how long to wait before retrying a 429 
# MAX_RETRIES
RETRY_SLEEP_SECONDS = 5
MAX_RETRIES         = 3

print(f"Catalog : {CATALOG}")
print(f"Schema  : {SCHEMA}")
print(f"Volume  : {VOLUME}")
print(f"Model   : {LLM_MODEL}")

In [0]:
%run ./validator

> Read raw airports data

`airports.dat.txt` is the OpenFlights dataset — 14,000 airports, no header row.  
We define the schema manually because there is no header to infer from.

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# Read Raw table

AIRPORTS_SCHEMA = StructType([
    StructField("airport_id",  IntegerType(), True),
    StructField("name",        StringType(),  True),
    StructField("city",        StringType(),  True),  
    StructField("country",     StringType(),  True),  
    StructField("iata",        StringType(),  True),
    StructField("icao",        StringType(),  True),
    StructField("latitude",    DoubleType(),  True),
    StructField("longitude",   DoubleType(),  True),
    StructField("altitude",    IntegerType(), True),
    StructField("timezone",    StringType(),  True),
    StructField("dst",         StringType(),  True),
    StructField("tz_database", StringType(),  True),
    StructField("type",        StringType(),  True),
    StructField("source",      StringType(),  True),
])

# Write raw table 
raw_df = (
    spark.read
    .format("csv")
    .option("header", "false")
    .option("quote", '"')
    .option("escape", '"')
    .schema(AIRPORTS_SCHEMA)
    .load(AIRPORTS_FILE)
    .withColumn("_loaded_at", F.current_timestamp())
)


> Load reference table

`cities500.txt` is the GeoNames dataset of all cities with population > 500.  
Tab-separated, 19 columns. We only need `name` (city) and `country_code`.  
A small random sample is passed into each LLM prompt to ground it in real geography.

In [0]:
GEONAMES_SCHEMA = StructType([
    StructField("geonameid",   IntegerType(), True),
    StructField("name",        StringType(),  True), 
    StructField("asciiname",   StringType(),  True),
    StructField("alternatenames", StringType(), True),
    StructField("latitude",    DoubleType(),  True),
    StructField("longitude",   DoubleType(),  True),
    StructField("feature_class", StringType(), True),
    StructField("feature_code", StringType(),  True),
    StructField("country_code", StringType(),  True), 
    StructField("cc2",         StringType(),  True),
    StructField("admin1",      StringType(),  True),
    StructField("admin2",      StringType(),  True),
    StructField("admin3",      StringType(),  True),
    StructField("admin4",      StringType(),  True),
    StructField("population",  StringType(),  True),
    StructField("elevation",   StringType(),  True),
    StructField("dem",         StringType(),  True),
    StructField("timezone",    StringType(),  True),
    StructField("modified",    StringType(),  True),
])

ref_df = (
    spark.read
    .format("csv")
    .option("header", "false")
    .option("sep", "\t")
    .schema(GEONAMES_SCHEMA)
    .load(REFERENCE_FILE)
)


> Create a sample ref for the LLM

In [0]:
city_reference = (
    ref_df
    .select("name")
    .dropna()
    .sample(False, 0.1, seed=42)
    .limit(REFERENCE_SAMPLE_SIZE)
    .toPandas()["name"]
    .tolist()
)

country_reference = (
    ref_df
    .select("country_code")
    .dropna()
    .distinct()
    .limit(REFERENCE_SAMPLE_SIZE)
    .toPandas()["country_code"]
    .tolist()
)

print(f"City reference sample   : {city_reference[:5]} ...")
print(f"Country reference sample: {country_reference[:5]} ...")

In [0]:
# These variables are serialised by Spark and sent to every worker node.
_TOKEN          = token
_WS_URL         = workspace_url.rstrip("/")
_MODEL          = LLM_MODEL
_SYSTEM_PROMPT  = _SYSTEM   # from %run ./validator
_USER_PROMPT    = _USER     # from %run ./validator
_RETRY_SLEEP    = RETRY_SLEEP_SECONDS
_MAX_RETRIES    = MAX_RETRIES


def make_udf(field_type: str, reference: list):
    """

    Why a factory? We need separate UDFs for city and country
    (different field_type and reference list). The factory avoids
    copy-pasting the same UDF code twice.

    Args:
        field_type : 'city' or 'country' — embedded in the LLM prompt
        reference  : list of clean example values from GeoNames

    Returns:
        A registered Spark UDF with returnType=RESULT_SCHEMA
    """
    # Copy each value into a local variable — guarantees clean closure capture.
    tok     = _TOKEN
    wsurl   = _WS_URL
    model   = _MODEL
    sys_p   = _SYSTEM_PROMPT
    user_p  = _USER_PROMPT
    ftype   = field_type
    ref     = list(reference)   
    sleep_s = _RETRY_SLEEP
    max_ret = _MAX_RETRIES

    @udf(returnType=RESULT_SCHEMA)
    def _udf(raw_value):
        """
        This function runs on Spark worker nodes — one call per row.
        Everything used inside must be serialisable (strings, lists, ints).
        Imports are placed inside the UDF because workers may not have
        the same imported modules as the driver session.
        """
        import json
        import time
        import requests

        # Helper: build the return tuple matching RESULT_SCHEMA field order
        def _ok(raw, corrected, conf, flag, reason):
            return (raw, corrected, conf, flag, reason)

        # Fast exit for null/empty values 
        if not raw_value or str(raw_value).strip() == "":
            return _ok(raw_value, raw_value, 1.0, "CLEAN", "Empty — skipped")

        # Fill prompt template with actual row data
        prompt = user_p.format(
            field_type = ftype,
            raw_value  = str(raw_value).strip(),
            reference  = json.dumps(ref[:15]),
        )

        # ── LLM call with retry on 429 (rate limit) 
        url = f"{wsurl}/serving-endpoints/{model}/invocations"

        for attempt in range(max_ret):
            try:
                resp = requests.post(
                    url,
                    headers={
                        "Authorization": f"Bearer {tok}",
                        "Content-Type":  "application/json",
                    },
                    json={
                        "messages": [
                            {"role": "system", "content": sys_p},
                            {"role": "user",   "content": prompt},
                        ],
                        "max_tokens":  200,
                        "temperature": 0.0,   # deterministic output
                    },
                    timeout=30,
                )

                # 429 = Too Many Requests — back off and retry
                if resp.status_code == 429:
                    time.sleep(sleep_s * (attempt + 1))  # exponential backoff: 5s, 10s, 15s
                    continue

                resp.raise_for_status() 

                # Parse LLM response
                text = resp.json()["choices"][0]["message"]["content"].strip()

                # Strip them , some models add them despite instructions
                if "```" in text:
                    text = text.split("```")[1]
                    if text.startswith("json"):
                        text = text[4:]
                text = text.strip()

                # Parse JSON and extract fields with safe defaults
                p = json.loads(text)
                return _ok(
                    raw_value,
                    str(p.get("corrected_value", raw_value)),
                    float(p.get("confidence_score", 0.0)),
                    str(p.get("flag",   "MANUAL")),
                    str(p.get("reason", "")),
                )

            except Exception as e:
                # Any non-retryable error ,flag as MANUAL, Row goes to silver_quarantine for human review
                return _ok(raw_value, raw_value, 0.0, "MANUAL",
                           f"Error: {str(e)[:120]}")

        return _ok(raw_value, raw_value, 0.0, "MANUAL",
                   f"Rate limit exceeded after {max_ret} retries")

    return _udf


city_udf    = make_udf("city",    city_reference)
country_udf = make_udf("country", country_reference)

print("UDFs registered — ready to validate")

# Quick smoke test 

In [0]:
# 5-row smoke test — checks the full UDF chain without touching all data
test_df = raw_df.limit(5)

test_result = (
    test_df
    .withColumn("city_result", city_udf(F.col("city")))
)

test_result.select(
    "city",
    F.col("city_result.corrected_value").alias("corrected"),
    F.col("city_result.confidence_score").alias("confidence"),
    F.col("city_result.flag").alias("flag"),
    F.col("city_result.reason").alias("reason"),
).show(5, truncate=False)


# If all rows show flag=MANUAL with reason='Error: ...'. stop and debug
# If you see CLEAN/FUZZY with real reasons → proceed to full validation

In [0]:
validated_df = (
    raw_df
    .withColumn("city_result",    city_udf(F.col("city")))
    .withColumn("country_result", country_udf(F.col("country")))

    
    # Pull the worst (lowest) confidence across both columns for routing
    .withColumn("_min_conf",
        F.least(
            F.col("city_result.confidence_score"),
            F.col("country_result.confidence_score"),
        )
    )
)

validated_df.select(
    "city",
    F.col("city_result.corrected_value"),
    F.col("city_result.confidence_score"),
    F.col("city_result.flag"),
    F.col("city_result.reason")
).show(20, truncate=False)

> Split and write silver into 3 tables

In [0]:
# BASE_PATH = f"/Volumes/{CATALOG}/raw/geo_assist/silver_tables"

# # Confirm path is accessible
# dbutils.fs.mkdirs(BASE_PATH)
# print(f"Writing Silver tables to: {BASE_PATH}")

# for layer, df, table_name in [
#     ("silver_clean",      clean_df,      "silver_clean"),
#     ("silver_review",     review_df,     "silver_review"),
#     ("silver_quarantine", quarantine_df, "silver_quarantine"),
# ]:
#     full_table = f"{CATALOG}.{SCHEMA}.{table_name}"
#     path       = f"{BASE_PATH}/{table_name}"

#     # Step 1 — write Delta files to Volume path
#     (df.drop("_min_conf")
#        .write
#        .format("delta")
#        .mode("overwrite")
#        .save(path))
#     print(f"  ✓ Written: {path}")

# validated_df.unpersist()
# raw_df.unpersist()
# print("\nDone — run 02_review.py next.")